# Brain MRI (BraTS) — nnU-Net training (Google Colab)

Colab port of [`notebooks/kaggle/03_brain_mri_nnunet_training.ipynb`](../kaggle/03_brain_mri_nnunet_training.ipynb).
**Read that notebook's compute-cost warning first** — it applies here
unchanged: a full nnU-Net BraTS run (5-fold, ~1000 epochs/fold) is a
multi-day, multi-GPU job. Colab free tier (T4, session limits, idle
disconnects) is even less suited to a full run than Kaggle's 12h sessions —
this notebook is a **pipeline smoke test on one fold with a reduced epoch
budget**, same as the Kaggle version.

**The one thing that genuinely differs from Kaggle here: getting BraTS onto
the machine.** Colab has no "Add Data" dataset mount. Pick one:

- **Option A — Kaggle API (recommended if you don't already have BraTS
  somewhere):** upload a Kaggle API token and download a BraTS mirror dataset
  the same way you'd add it on Kaggle, just via the API instead of the UI.
- **Option B — Google Drive:** if you've already downloaded BraTS (e.g., via
  Synapse, which requires its own registration/data-use agreement), upload it
  to Drive once and point `BRATS_SOURCE_DIR` at it.

## Colab setup
1. **Runtime → Change runtime type → T4 GPU** (or better — 3D patch training
   is memory-hungry; a Colab Pro A100 finishes noticeably faster if you have it).
2. Mount Drive (cell 2) for persistent storage of the final checkpoint —
   nnU-Net's raw/preprocessed intermediate files stay on the (faster, but
   ephemeral) local Colab disk since they're large and re-derivable.

In [ ]:
# Cell 1 — install nnU-Net v2.
!pip install -q nnunetv2

In [ ]:
# Cell 2 — mount Drive (final results only; nnU-Net's raw/preprocessed working
# files are large and rederivable, so they stay on local Colab disk).
from google.colab import drive
drive.mount("/content/drive")

import os
DRIVE_RESULTS_DIR = "/content/drive/MyDrive/aegis-dx/brain-mri-nnunet/results"
os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)

## Option A — get BraTS via the Kaggle API

1. Go to [kaggle.com/settings](https://www.kaggle.com/settings) → *API* →
   **Create New Token**. This downloads a `kaggle.json` file.
2. Run the cell below and upload that file when prompted.
3. Search [kaggle.com/datasets](https://www.kaggle.com/datasets) for **"BraTS"**,
   open a mirror dataset's page, click the **⋮ menu → Copy API command**, and
   paste the resulting `kaggle datasets download -d <owner>/<slug>` into
   `KAGGLE_DATASET_SLUG` below — Kaggle dataset slugs for BraTS mirrors change
   over time, so grab the exact one from the dataset page rather than guessing.

In [ ]:
# Cell 3 — Option A: Kaggle API download. Skip this cell entirely if you're
# using Option B (Drive) instead.
from google.colab import files

os.makedirs("/root/.kaggle", exist_ok=True)
uploaded = files.upload()  # select your kaggle.json when prompted
!mv kaggle.json /root/.kaggle/kaggle.json
!chmod 600 /root/.kaggle/kaggle.json

# Fill this in from the dataset page's "Copy API command" (see markdown above).
KAGGLE_DATASET_SLUG = "awsaf49/brats20-dataset-training-validation"  # example - verify this still exists and matches BraTS's expected layout before trusting it

!pip install -q kaggle
!kaggle datasets download -d {KAGGLE_DATASET_SLUG} -p /content/brats_download --unzip
BRATS_SOURCE_DIR = "/content/brats_download"
print("Downloaded to:", BRATS_SOURCE_DIR)

## Option B — already have BraTS in Drive

If you downloaded BraTS yourself (e.g., from Synapse, after its own
registration/data-use agreement) and uploaded it to Drive, skip cell 3 above
and just point this cell at that folder instead.

In [ ]:
# Cell 4 — Option B: point at BraTS already sitting in Drive. Only run this
# (instead of cell 3) if you're using your own copy.
# BRATS_SOURCE_DIR = "/content/drive/MyDrive/aegis-dx/brats-raw"
print("BRATS_SOURCE_DIR is set to:", BRATS_SOURCE_DIR)

In [ ]:
# Cell 5 — nnU-Net environment variables. Working files on local disk (fast,
# ephemeral); only the final results get copied to Drive at the end.
import glob
import json
import shutil

import nibabel as nib
import numpy as np

NNUNET_BASE = "/content/nnunet"
os.environ["nnUNet_raw"] = f"{NNUNET_BASE}/nnUNet_raw"
os.environ["nnUNet_preprocessed"] = f"{NNUNET_BASE}/nnUNet_preprocessed"
os.environ["nnUNet_results"] = f"{NNUNET_BASE}/nnUNet_results"

for path in os.environ["nnUNet_raw"], os.environ["nnUNet_preprocessed"], os.environ["nnUNet_results"]:
    os.makedirs(path, exist_ok=True)

DATASET_ID = 1
DATASET_NAME = f"Dataset{DATASET_ID:03d}_BraTS"
DATASET_DIR = os.path.join(os.environ["nnUNet_raw"], DATASET_NAME)
os.makedirs(os.path.join(DATASET_DIR, "imagesTr"), exist_ok=True)
os.makedirs(os.path.join(DATASET_DIR, "labelsTr"), exist_ok=True)
print("nnU-Net dataset dir:", DATASET_DIR)

In [ ]:
# Cell 6 — find BraTS case folders (searches BRATS_SOURCE_DIR, not /kaggle/input).
MODALITY_KEYWORDS = {"t1": 0, "t1ce": 1, "t2": 2, "flair": 3}

seg_files = glob.glob(f"{BRATS_SOURCE_DIR}/**/*seg.nii.gz", recursive=True) + glob.glob(
    f"{BRATS_SOURCE_DIR}/**/*seg.nii", recursive=True
)
if not seg_files:
    raise FileNotFoundError(
        f"No BraTS segmentation files found under {BRATS_SOURCE_DIR}. "
        "Check the Kaggle download in cell 3 succeeded, or that BRATS_SOURCE_DIR "
        "in cell 4 points at the right Drive folder."
    )
case_dirs = sorted({os.path.dirname(path) for path in seg_files})
print(f"Found {len(case_dirs)} BraTS case folders. Example: {case_dirs[0]}")

MAX_CASES = 60  # Kaggle-session-equivalent scope for a pipeline smoke test
case_dirs = case_dirs[:MAX_CASES]
print(f"Using {len(case_dirs)} cases for this run.")

In [ ]:
# Cell 7 — convert to nnU-Net's expected layout (identical logic to Kaggle).
def find_modality_file(case_dir: str, keyword: str) -> str | None:
    matches = [
        f for f in glob.glob(os.path.join(case_dir, "*.nii*"))
        if keyword in os.path.basename(f).lower() and "seg" not in os.path.basename(f).lower()
    ]
    return matches[0] if matches else None


def remap_brats_labels(segmentation: np.ndarray) -> np.ndarray:
    remapped = segmentation.copy()
    remapped[segmentation == 4] = 3
    return remapped


converted_case_ids = []
for case_dir in case_dirs:
    case_id = os.path.basename(case_dir.rstrip("/"))
    modality_paths = {keyword: find_modality_file(case_dir, keyword) for keyword in MODALITY_KEYWORDS}
    if any(path is None for path in modality_paths.values()):
        print(f"skip {case_id}: missing a modality file")
        continue

    seg_path = glob.glob(os.path.join(case_dir, "*seg.nii*"))[0]

    for keyword, channel_index in MODALITY_KEYWORDS.items():
        destination = os.path.join(DATASET_DIR, "imagesTr", f"{case_id}_{channel_index:04d}.nii.gz")
        shutil.copy(modality_paths[keyword], destination)

    seg_image = nib.load(seg_path)
    seg_data = remap_brats_labels(np.asarray(seg_image.dataobj))
    remapped_image = nib.Nifti1Image(seg_data.astype(np.uint8), seg_image.affine, seg_image.header)
    nib.save(remapped_image, os.path.join(DATASET_DIR, "labelsTr", f"{case_id}.nii.gz"))

    converted_case_ids.append(case_id)

print(f"Converted {len(converted_case_ids)} cases.")

In [ ]:
# Cell 8 — dataset.json manifest (identical to Kaggle).
dataset_json = {
    "channel_names": {"0": "T1", "1": "T1ce", "2": "T2", "3": "FLAIR"},
    "labels": {"background": 0, "necrotic_core": 1, "edema": 2, "enhancing_tumor": 3},
    "numTraining": len(converted_case_ids),
    "file_ending": ".nii.gz",
}
with open(os.path.join(DATASET_DIR, "dataset.json"), "w") as handle:
    json.dump(dataset_json, handle, indent=2)
print(json.dumps(dataset_json, indent=2))

In [ ]:
# Cell 9 — plan & preprocess.
!nnUNetv2_plan_and_preprocess -d {DATASET_ID} --verify_dataset_integrity -c 3d_fullres

## Train — one fold, reduced epoch budget

Same reduced-epoch trainer variant as Kaggle, for the same reason: a pipeline
smoke test, not a benchmark run. **Colab's idle-disconnect timeout is stricter
than Kaggle's** — keep the browser tab open/active, or use Colab Pro's
background execution if this run risks exceeding ~90 minutes of apparent
inactivity.

In [ ]:
# Cell 10 — train fold 0 with the reduced-epoch trainer.
!nnUNetv2_train {DATASET_ID} 3d_fullres 0 -tr nnUNetTrainer_50epochs

In [ ]:
# Cell 11 — inference smoke test on one case (same caveat as Kaggle: this
# reuses a training case, not a real held-out set).
INFERENCE_INPUT_DIR = f"{NNUNET_BASE}/inference_input"
INFERENCE_OUTPUT_DIR = f"{NNUNET_BASE}/inference_output"
os.makedirs(INFERENCE_INPUT_DIR, exist_ok=True)
os.makedirs(INFERENCE_OUTPUT_DIR, exist_ok=True)

sample_case_id = converted_case_ids[0]
for channel_index in range(4):
    source = os.path.join(DATASET_DIR, "imagesTr", f"{sample_case_id}_{channel_index:04d}.nii.gz")
    shutil.copy(source, INFERENCE_INPUT_DIR)

!nnUNetv2_predict -i {INFERENCE_INPUT_DIR} -o {INFERENCE_OUTPUT_DIR} -d {DATASET_ID} -c 3d_fullres -f 0 -tr nnUNetTrainer_50epochs
print("Predicted files:", os.listdir(INFERENCE_OUTPUT_DIR))

In [ ]:
# Cell 12 — copy final results (checkpoint + summary) to Drive for persistence.
shutil.copytree(os.environ["nnUNet_results"], DRIVE_RESULTS_DIR, dirs_exist_ok=True)

summary_paths = glob.glob(f"{DRIVE_RESULTS_DIR}/**/summary.json", recursive=True)
if summary_paths:
    with open(summary_paths[0]) as handle:
        summary = json.load(handle)
    print("Mean Dice per class:", summary.get("mean", {}))
print("Results copied to:", DRIVE_RESULTS_DIR)

## Next steps

Identical to the Kaggle version: this is a pipeline smoke test, not a trained
model. Scale to the full case set, default trainer (1000 epochs), and 5 folds
on a multi-GPU box before comparing against published nnU-Net BraTS Dice
baselines. See `notebooks/kaggle/03_brain_mri_nnunet_training.ipynb`'s final
cell for the full list, including wiring MedSAM/MedSAM2 for interactive
refinement.